# Vantage — UMAP Embedding on Colab

**What this notebook does:**
1. Installs dependencies
2. Uploads `vantage.duckdb` + `tfidf.pkl` from your local machine
3. Runs UMAP on the 2,117 NSW suburb TF-IDF vectors
4. Writes `reducer.pkl` + updates `umap_coords` table in `vantage.duckdb`
5. Downloads both artifacts back to your machine

**Runtime:** Use GPU (T4) or High-RAM CPU — either is fine for 2K samples.

In [ ]:
# Cell 1 — Install
!pip install umap-learn scikit-learn duckdb joblib tqdm pyarrow -q

In [ ]:
# Cell 2 — Upload files from your local machine
from google.colab import files

print('Upload backend/vantage.duckdb  (the ~13 MB database)')
uploaded = files.upload()   # select backend/vantage.duckdb

print('Upload backend/tfidf.pkl  (the ~1.4 KB vectorizer)')
uploaded2 = files.upload()  # select backend/tfidf.pkl

print('Done uploading.')

In [ ]:
# Cell 3 — Run UMAP embedding
import duckdb
import joblib
import numpy as np
from sklearn.decomposition import TruncatedSVD
from umap import UMAP

DB_PATH      = 'vantage.duckdb'
TFIDF_PATH   = 'tfidf.pkl'
REDUCER_PATH = 'reducer.pkl'

# --- load vectorizer ---
vectorizer = joblib.load(TFIDF_PATH)
print(f'Vectorizer loaded — {len(vectorizer.vocabulary_)} features')

# --- build suburb corpus ---
con = duckdb.connect(DB_PATH)
suburb_docs = con.execute("""
    SELECT h3_r7, string_agg(category_label, ' ') AS doc
    FROM venues
    WHERE category_label IS NOT NULL AND is_closed = false
    GROUP BY h3_r7
    HAVING count(*) >= 5
    ORDER BY h3_r7
""").fetchall()

cell_ids = [r[0] for r in suburb_docs]
corpus   = [r[1] for r in suburb_docs]
print(f'Suburbs to embed: {len(cell_ids)}')

# --- TF-IDF transform ---
X = vectorizer.transform(corpus)
n_samples, n_dims = X.shape
print(f'TF-IDF matrix: {n_samples} × {n_dims}')

# --- PCA pre-reduction ---
if n_dims > 100:
    print(f'TruncatedSVD: {n_dims} → 100 dims …')
    svd = TruncatedSVD(n_components=100, random_state=42)
    X_dense = svd.fit_transform(X)
    print(f'SVD variance explained: {svd.explained_variance_ratio_.sum()*100:.1f}%')
else:
    X_dense = X.toarray() if hasattr(X, 'toarray') else np.array(X)

# --- UMAP fit ---
n_neighbors = min(15, n_samples - 1)
print(f'Fitting UMAP (n_neighbors={n_neighbors}, metric=cosine) …')
reducer = UMAP(
    n_components=2,
    n_neighbors=n_neighbors,
    min_dist=0.1,
    metric='cosine',
    random_state=42,
    low_memory=True,
    verbose=True,
)
coords = reducer.fit_transform(X_dense)
print(f'UMAP done — output shape: {coords.shape}')

In [ ]:
# Cell 4 — Save reducer.pkl + update umap_coords in DB

# save reducer
joblib.dump(reducer, REDUCER_PATH)
import os
print(f'reducer.pkl saved ({os.path.getsize(REDUCER_PATH)/1024:.1f} KB)')

# update umap_coords in duckdb
con.execute("""
    CREATE OR REPLACE TABLE umap_coords (
        h3_r7  VARCHAR PRIMARY KEY,
        umap_x DOUBLE,
        umap_y DOUBLE
    )
""")
rows = [
    (cell_ids[i], float(coords[i, 0]), float(coords[i, 1]))
    for i in range(len(cell_ids))
]
con.executemany('INSERT INTO umap_coords VALUES (?, ?, ?)', rows)
n_rows = con.execute('SELECT count(*) FROM umap_coords').fetchone()[0]
print(f'umap_coords updated — {n_rows} rows')
con.close()

# quick sanity check
print(f'X range: [{coords[:,0].min():.2f}, {coords[:,0].max():.2f}] × [{coords[:,1].min():.2f}, {coords[:,1].max():.2f}]')

In [ ]:
# Cell 5 — Download both artifacts
from google.colab import files

print('Downloading reducer.pkl …')
files.download(REDUCER_PATH)

print('Downloading vantage.duckdb (with umap_coords) …')
files.download(DB_PATH)

print()
print('Done! Copy both files into your local backend/ directory:')
print('  backend/reducer.pkl')
print('  backend/vantage.duckdb')